In [1]:
import json
from pathlib import Path

import polars as pl

ROOT = Path("../data/raw/so101_pickplace")
info = json.loads((ROOT / "meta" / "info.json").read_text())

print("format:", info["codebase_version"])
print("fps:", info["fps"])
print("episodes:", info["total_episodes"], "| frames:", info["total_frames"])

format: v3.0
fps: 30
episodes: 50 | frames: 11939


In [2]:
for name, spec in info["features"].items():
    print(f"{name:35s} {spec['dtype']:10s} {spec['shape']}")

action                              float32    [6]
observation.state                   float32    [6]
observation.images.up               video      [480, 640, 3]
observation.images.side             video      [480, 640, 3]
timestamp                           float32    [1]
frame_index                         int64      [1]
episode_index                       int64      [1]
index                               int64      [1]
task_index                          int64      [1]


In [3]:
df = pl.scan_parquet(f"{ROOT}/data/*/*.parquet").collect()
print(df.shape)
df.head()

(11939, 7)


action,observation.state,timestamp,frame_index,episode_index,index,task_index
list[f32],list[f32],f32,i64,i64,i64,i64
"[1.911765, -99.41053, … 1.622875]","[1.956088, -98.743721, … 1.409396]",0.0,0,0,0,0
"[1.911765, -99.41053, … 1.622875]","[1.956088, -98.743721, … 1.409396]",0.033333,1,0,1,0
"[1.911765, -99.41053, … 1.622875]","[1.956088, -98.743721, … 1.409396]",0.066667,2,0,2,0
"[1.911765, -99.41053, … 1.622875]","[1.956088, -98.743721, … 1.409396]",0.1,3,0,3,0
"[1.911765, -99.41053, … 1.622875]","[1.956088, -98.743721, … 1.409396]",0.133333,4,0,4,0


In [4]:
episodes = (
    df.group_by("episode_index")
    .agg(
        pl.len().alias("frames"),
        pl.col("timestamp").min().alias("t_start"),
        pl.col("timestamp").max().alias("t_end"),
    )
    .with_columns((pl.col("frames") / info["fps"]).alias("duration_s"))
    .sort("episode_index")
)
episodes.describe()

statistic,episode_index,frames,t_start,t_end,duration_s
str,f64,f64,f64,f64,f64
"""count""",50.0,50.0,50.0,50.0,50.0
"""null_count""",0.0,0.0,0.0,0.0,0.0
"""mean""",24.5,238.78,0.0,7.926,7.959333
"""std""",14.57738,30.795103,0.0,1.026503,1.026503
"""min""",0.0,183.0,0.0,6.066667,6.1
"""25%""",12.0,215.0,0.0,7.133333,7.166667
"""50%""",25.0,231.0,0.0,7.666667,7.7
"""75%""",37.0,262.0,0.0,8.7,8.733333
"""max""",49.0,306.0,0.0,10.166667,10.2


In [5]:
expected_dt = 1 / info["fps"]

timed = (
    df.sort(["episode_index", "frame_index"])
    .with_columns(pl.col("timestamp").diff().over("episode_index").alias("dt"))
)

print("expected dt:", expected_dt)
print(timed["dt"].describe())

bad = timed.filter((pl.col("dt") - expected_dt).abs() > 1e-3)
print("frames with irregular spacing:", bad.height)

expected dt: 0.03333333333333333
shape: (9, 2)
┌────────────┬───────────┐
│ statistic  ┆ value     │
│ ---        ┆ ---       │
│ str        ┆ f64       │
╞════════════╪═══════════╡
│ count      ┆ 11889.0   │
│ null_count ┆ 50.0      │
│ mean       ┆ 0.033333  │
│ std        ┆ 1.4033e-7 │
│ min        ┆ 0.033333  │
│ 25%        ┆ 0.033333  │
│ 50%        ┆ 0.033333  │
│ 75%        ┆ 0.033333  │
│ max        ┆ 0.033334  │
└────────────┴───────────┘
frames with irregular spacing: 0


In [6]:
import duckdb

duckdb.sql(f"""
    SELECT episode_index,
           COUNT(*)       AS frames,
           MAX(timestamp) AS duration_s
    FROM '{ROOT}/data/*/*.parquet'
    GROUP BY episode_index
    ORDER BY frames DESC
    LIMIT 5
""")

episode_indexBIGINT,framesBIGINT,duration_sFLOAT
41,306,10.166666984558105
46,303,10.066666603088379
0,303,10.066666603088379
40,293,9.733333587646484
30,288,9.566666603088379


In [7]:
ep0 = df.filter(pl.col("episode_index") == 0).sort("frame_index")
ep0.select(
    "frame_index",
    pl.col("action").list.get(5).alias("gripper_commanded"),
    pl.col("observation.state").list.get(5).alias("gripper_measured"),
).slice(0, 15)

frame_index,gripper_commanded,gripper_measured
i64,f32,f32
0,1.622875,1.409396
1,1.622875,1.409396
2,1.622875,1.409396
3,1.622875,1.409396
4,1.622875,1.409396
…,…,…
10,1.622875,1.409396
11,1.622875,1.409396
12,1.622875,1.409396


In [8]:
moves = ep0.with_columns(
    pl.col("action").list.get(5).diff().abs().alias("cmd_change"),
    pl.col("observation.state").list.get(5).diff().abs().alias("meas_change"),
)

first_cmd = moves.filter(pl.col("cmd_change") > 0.5)["frame_index"].min()
first_meas = moves.filter(pl.col("meas_change") > 0.5)["frame_index"].min()

print("command first changes at frame:", first_cmd)
print("measured first changes at frame:", first_meas)
print("lag (frames):", first_meas - first_cmd, "→", (first_meas - first_cmd) / info["fps"], "seconds")

command first changes at frame: 92
measured first changes at frame: 99
lag (frames): 7 → 0.23333333333333334 seconds


In [9]:
info["features"]["observation.state"]["names"]

['shoulder_pan.pos',
 'shoulder_lift.pos',
 'elbow_flex.pos',
 'wrist_flex.pos',
 'wrist_roll.pos',
 'gripper.pos']

In [10]:
print(info["features"]["observation.state"].get("names"))
print(info["features"]["action"].get("names"))

['shoulder_pan.pos', 'shoulder_lift.pos', 'elbow_flex.pos', 'wrist_flex.pos', 'wrist_roll.pos', 'gripper.pos']
['shoulder_pan.pos', 'shoulder_lift.pos', 'elbow_flex.pos', 'wrist_flex.pos', 'wrist_roll.pos', 'gripper.pos']
